In [36]:
!pip install earthengine-api
!pip install geemap -q
!pip install rasterio -q
!pip install matplotlib -q
!pip install numpy -q
!pip install jupyterlab -q
!pip install mss -q


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
import ee
import geemap

ee.Authenticate()
ee.Initialize()

In [38]:
roi = ee.FeatureCollection('FAO/GAUL/2015/level1') \
    .filter(ee.Filter.eq('ADM0_NAME','Thailand')) \
    .filter(ee.Filter.eq('ADM1_NAME','Sukhothai'))

In [39]:
dem = ee.Image('USGS/SRTMGL1_003').clip(roi)

slope = ee.Terrain.slope(dem)

In [40]:
rain = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
    .filterDate('2023-01-01','2023-12-31') \
    .select('precipitation') \
    .sum() \
    .clip(roi)

In [41]:
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(roi) \
    .filterDate('2023-01-01','2023-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',20)) \
    .median() \
    .clip(roi)

ndvi = s2.normalizedDifference(['B8','B4'])

In [42]:
Map = geemap.Map(center=[17,99.7], zoom=8)

Map.addLayer(
    ndvi,
    {'min':0,'max':1,'palette':['white','green']},
    "NDVI"
)

Map

Map(center=[17, 99.7], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

In [43]:
water = ee.Image('JRC/GSW1_4/GlobalSurfaceWater') \
    .select('occurrence') \
    .clip(roi)

river = water.gt(50).selfMask()

dist_river = river.fastDistanceTransform(30).sqrt().multiply(30)

In [44]:
def normalize(img):

    stats = img.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=roi,
        scale=1000,
        maxPixels=1e13
    )

    band = img.bandNames().get(0)

    min_val = ee.Number(stats.get(ee.String(band).cat('_min')))
    max_val = ee.Number(stats.get(ee.String(band).cat('_max')))

    return img.subtract(min_val).divide(max_val.subtract(min_val))

In [45]:
rain_n = normalize(rain)

elev_n = ee.Image(1).subtract(normalize(dem))

slope_n = ee.Image(1).subtract(normalize(slope))

river_n = ee.Image(1).subtract(normalize(dist_river))

ndvi_n = ee.Image(1).subtract(normalize(ndvi))

In [46]:
flood_risk = (
    rain_n.multiply(0.25)
    .add(elev_n.multiply(0.20))
    .add(slope_n.multiply(0.20))
    .add(river_n.multiply(0.20))
    .add(ndvi_n.multiply(0.15))
)

In [47]:
Map = geemap.Map(center=[17,99.7], zoom=8)

Map.addLayer(
    flood_risk,
    {'min':0,'max':1,
     'palette':['green','yellow','orange','red']},
    "Flood Risk"
)

Map

Map(center=[17, 99.7], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

In [48]:
risk_class = flood_risk.expression(
    "(b(0) <= 0.25) ? 1"
    ": (b(0) <= 0.50) ? 2"
    ": (b(0) <= 0.75) ? 3"
    ": 4"
).clip(roi)

In [49]:
Map = geemap.Map(center=[17, 99.7], zoom=8)

vis_params = {
    'min': 1,
    'max': 4,
    'palette': [
        '#2DC937',  # Low
        '#E7B416',  # Moderate
        '#CC3232',  # High
        '#7A0019'   # Very High
    ]
}

Map.addLayer(risk_class, vis_params, "Flood Risk Classes")

Map

Map(center=[17, 99.7], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

In [50]:
Map = geemap.Map(center=[17,99.7], zoom=8)

Map.addLayer(
    flood_risk,
    {'min':0,'max':1,'palette':['green','yellow','orange','red']},
    "Flood Risk Index"
)

Map.addLayer(
    risk_class,
    {'min':1,'max':4,'palette':['#2DC937','#E7B416','#CC3232','#7A0019']},
    "Flood Risk Classes"
)

Map

Map(center=[17, 99.7], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…